In [ ]:
from utils.params import *

### Merge tables and write to gold layer
*this layer removes "favorite_count" as this column generated errors and is not used in the analysis

In [ ]:
def list_delta_tables(base_path):
    """
    List all Delta tables at the second level under the base_path (e.g., silver/coffee/badges).
    """
    delta_tables = []
    try:
        # List first-level directories (e.g., coffee/)
        level1_items = dbutils.fs.ls(base_path)
        for level1 in level1_items:
            if level1.isDir():
                # List second-level directories (e.g., badges/, comments/)
                level2_items = dbutils.fs.ls(level1.path)
                for level2 in level2_items:
                    if level2.isDir():
                        delta_tables.append(level2.path.rstrip('/'))  # Remove trailing slash
    except Exception as e:
        print(f"Error accessing path {base_path}: {e}")
    return delta_tables

# Start the search from the silver directory
base_path = "/mnt/stackoverflow/silver/"
delta_table_paths = list_delta_tables(base_path)
print(f"Found Delta tables: {delta_table_paths}")

Writing to ./badges
Completed writing to ./badges

Writing to ./comments
Completed writing to ./comments

Writing to ./posthistory
Completed writing to ./posthistory

Writing to ./posts
Completed writing to ./posts

Writing to ./tags
Completed writing to ./tags

Writing to ./users
Completed writing to ./users

Writing to ./votes
Completed writing to ./votes



### Process each table name and merge Delta tables

In [ ]:
for tablename in TABLE_NAMES:
    merged_df = None
    print(f"Writing to ./{tablename}")
    for table_path in delta_table_paths:
        if table_path.rstrip('/').split('/')[-1].lower() == tablename.lower():
            try:
                # Read the Delta table
                df = spark.read.format("delta").load(table_path)
                # Drop 'favorite_count' column if it exists
                for col in df.columns:
                    if 'favorite_count' in col:
                        df = df.drop(col)
                # If merged_df is empty, assign the first DataFrame to it
                if merged_df is None:
                    merged_df = df
                else:
                    # Union (merge) the DataFrames using unionByName
                    merged_df = merged_df.unionByName(df, allowMissingColumns=True)
            except Exception as e:
                print(f"Error processing Delta table {table_path}: {e}")
                continue
    
    if merged_df is not None:
        # Define the destination path for the gold layer
        dest = f'/mnt/stackoverflow/gold/{tablename}/'
        # Write the merged DataFrame as a Delta table, overwriting existing data
        merged_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(dest)
        print(f"Completed writing to ./{tablename}")
    else:
        print(f"No data found for {tablename}, skipping write operation.")
    print('')